# Lesson 03 — Exercise Solutions

Worked answers to the five exercises at the end of
[`03_regularization.ipynb`](03_regularization.ipynb). Two of them, early stopping and the
data augmentation identity, reveal that regularisation is happening in places you would
not have looked for it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from regularization import (
    polynomial_features, zscore_normalize, train_test_split, k_fold_indices,
    compute_cost_linear, compute_gradient_linear, ridge_normal_equation,
    lasso_gradient_descent, gradient_descent,
)

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True

# the dataset from the lesson, rebuilt exactly
def truth(x):
    return np.sin(2.5 * x)

NOISE = 0.25
m = 40
rng = np.random.default_rng(1)
x_all = np.sort(rng.uniform(-1, 1, m))
y_all = truth(x_all) + rng.normal(0, NOISE, m)
order = np.random.default_rng(2).permutation(m)
idx_test, idx_train = order[:20], order[20:]
x_train, y_train = x_all[idx_train], y_all[idx_train]
x_test, y_test = x_all[idx_test], y_all[idx_test]


def fit_polynomial(degree, lam=0.0, x_fit=None, y_fit=None):
    # returns a prediction function plus the fitted weights
    x_fit = x_train if x_fit is None else x_fit
    y_fit = y_train if y_fit is None else y_fit
    X_scaled, mu, sigma = zscore_normalize(polynomial_features(x_fit, degree))
    w, b = ridge_normal_equation(X_scaled, y_fit, lam)

    def predict(x_new):
        X_new, _, _ = zscore_normalize(polynomial_features(x_new, degree), mu, sigma)
        return X_new @ w + b

    return predict, w, b


print(f"training on {len(y_train)} points, testing on {len(y_test)}")

---
# Exercise 1 — Ridge as data augmentation

> Ridge with penalty $\lambda$ is exactly ordinary least squares on an enlarged dataset:
> append $n$ extra rows equal to $\sqrt{\lambda}\,I$ with targets of zero. Verify this
> numerically and explain why the trick works.

## Why it works

Ordinary least squares minimises $\lVert X_b\theta - y \rVert^2$, where $X_b$ is the design
matrix with a leading column of ones and $\theta = (b, w_1, \ldots, w_n)$. Now stack $n$
extra rows underneath it, each one all zeros except for $\sqrt{\lambda}$ in the column of a
single weight, and give every extra row a target of zero:

$$X_{\text{aug}} = \begin{bmatrix} X_b \\[2pt] \mathbf{0} \;\big|\; \sqrt{\lambda}\,I_n \end{bmatrix}
\qquad
y_{\text{aug}} = \begin{bmatrix} y \\[2pt] \mathbf{0}_n \end{bmatrix}$$

A squared norm splits over stacked blocks, so the augmented objective separates into the
original one plus a contribution from the new rows:

$$\lVert X_{\text{aug}}\theta - y_{\text{aug}} \rVert^2
= \lVert X_b\theta - y \rVert^2 + \big\lVert \sqrt{\lambda}\,w - \mathbf{0} \big\rVert^2
= \lVert X_b\theta - y \rVert^2 + \lambda\lVert w \rVert^2$$

which is precisely the ridge objective. The leading column of zeros in the augmented block
is what exempts the bias from the penalty.

**In one sentence:** ridge is ordinary least squares plus $n$ invented examples, each one
insisting that a particular weight should be zero, with $\lambda$ setting how loudly they
insist.

In [ ]:
degree, lam = 8, 3.0
X_scaled, _, _ = zscore_normalize(polynomial_features(x_train, degree))
n_features = X_scaled.shape[1]
n_rows = X_scaled.shape[0]

X_b = np.hstack([np.ones((n_rows, 1)), X_scaled])                  # (20, 9)
extra_rows = np.hstack([np.zeros((n_features, 1)),                 # (8, 9)
                        np.sqrt(lam) * np.eye(n_features)])
X_aug = np.vstack([X_b, extra_rows])                               # (28, 9)
y_aug = np.concatenate([y_train, np.zeros(n_features)])            # (28,)

print(f"original design matrix {X_b.shape}, augmented {X_aug.shape}")

# plain least squares on the augmented data, using no penalty at all
theta_aug = np.linalg.solve(X_aug.T @ X_aug, X_aug.T @ y_aug)
w_aug, b_aug = theta_aug[1:], theta_aug[0]

# ridge on the original data
w_ridge, b_ridge = ridge_normal_equation(X_scaled, y_train, lam)

print(f"\nlargest difference in the weights : {np.max(np.abs(w_aug - w_ridge)):.3e}")
print(f"difference in the bias            : {abs(b_aug - b_ridge):.3e}")
print(f"identical to machine precision    : {np.allclose(w_aug, w_ridge) and np.isclose(b_aug, b_ridge)}")

The two agree exactly, not approximately. This is an identity, not an approximation.

Three things follow from it that are worth carrying away.

**It explains the conditioning fix.** Section 4 of the lesson showed the penalty rescuing a
system with more features than examples. The augmented view makes that obvious: adding $n$
rows containing $\sqrt{\lambda}I_n$ guarantees the stacked matrix has full column rank, no
matter how few real examples there are.

**It explains why the penalty is a prior.** Those invented rows say "in the absence of
evidence, each weight is zero". In Bayesian terms ridge is the maximum a posteriori
estimate under a Gaussian prior centred at zero, and $\lambda$ is the inverse of that
prior's variance. Lasso corresponds to a Laplace prior, whose sharp peak at zero is the
probabilistic version of the diamond corners.

**It is how some libraries implement it.** Rather than modifying the solver, they build the
augmented matrix and call the ordinary one.

---
# Exercise 2 — Early stopping is regularisation

> Fit the degree 12 model with $\lambda = 0$ by gradient descent, recording test error
> every 100 iterations. Show that it falls, bottoms out, and rises. Explain the connection
> to the weight norm.

In [ ]:
X_train_12, mu_12, sigma_12 = zscore_normalize(polynomial_features(x_train, 12))
X_test_12, _, _ = zscore_normalize(polynomial_features(x_test, 12), mu_12, sigma_12)

w, b = np.zeros(12), 0.0
iters, train_curve, test_curve, norm_curve = [], [], [], []

for i in range(30_001):
    if i % 100 == 0:
        iters.append(i)
        train_curve.append(float(np.mean((X_train_12 @ w + b - y_train) ** 2)))
        test_curve.append(float(np.mean((X_test_12 @ w + b - y_test) ** 2)))
        norm_curve.append(float(np.linalg.norm(w)))
    dj_dw, dj_db = compute_gradient_linear(X_train_12, y_train, w, b, lam=0.0)
    w = w - 0.05 * dj_dw
    b = b - 0.05 * dj_db

test_curve = np.array(test_curve)
best_step = int(np.argmin(test_curve))

print(f"no penalty at all, lambda = 0 throughout\n")
print(f"{'iteration':>12}{'train MSE':>12}{'test MSE':>12}{'||w||':>10}")
for k in (0, 10, 20, best_step, 100, 200, 300):
    tag = "  <-- best test error" if k == best_step else ""
    print(f"{iters[k]:>12}{train_curve[k]:>12.4f}{test_curve[k]:>12.4f}{norm_curve[k]:>10.3f}{tag}")

print(f"\nstopping at iteration {iters[best_step]} gives test MSE {test_curve[best_step]:.4f}")
print(f"running to the end gives          test MSE {test_curve[-1]:.4f}")
print(f"the best ridge fit from the lesson achieved      {0.0481:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].plot(iters, train_curve, label="training error")
axes[0].plot(iters, test_curve, label="test error")
axes[0].axvline(iters[best_step], color="tab:green", ls=":", label="best stopping point")
axes[0].set_xscale("symlog"); axes[0].set_yscale("log")
axes[0].set_xlabel("iteration"); axes[0].set_ylabel("MSE"); axes[0].legend(fontsize=8)
axes[0].set_title("Test error makes the same U shape")

axes[1].plot(iters, norm_curve, color="tab:red")
axes[1].axvline(iters[best_step], color="tab:green", ls=":")
axes[1].set_xscale("symlog"); axes[1].set_xlabel("iteration"); axes[1].set_ylabel("$\\|w\\|$")
axes[1].set_title("The weight norm only ever grows")

axes[2].plot(norm_curve, test_curve, lw=1.5)
axes[2].plot(norm_curve[best_step], test_curve[best_step], "go", ms=9, label="best")
axes[2].set_yscale("log"); axes[2].set_xlabel("$\\|w\\|$ reached so far")
axes[2].set_ylabel("test MSE"); axes[2].legend(fontsize=8)
axes[2].set_title("Test error against weight norm, the ridge curve in disguise")
plt.tight_layout(); plt.show()

## The connection

Gradient descent starts at $w = 0$ and grows the weights as it goes. The middle panel shows
$\|w\|$ climbing monotonically and never turning back. So **the number of iterations is
itself a budget on the weight norm.** Stopping after $t$ steps is roughly equivalent to
solving the problem with a constraint $\|w\| \leq c(t)$, which is exactly what ridge does
explicitly.

Read the panels together and the equivalence is hard to miss:

| | ridge | early stopping |
|---|---|---|
| the dial | $\lambda$, decreasing | iteration count, increasing |
| at one extreme | $\lambda \to \infty$ gives $w = 0$ | 0 iterations gives $w = 0$ |
| at the other | $\lambda = 0$ gives the unconstrained fit | many iterations gives the unconstrained fit |
| in between | a U shaped test error curve | a U shaped test error curve |

The third panel plots test error against the weight norm actually reached, which turns the
training run into something that looks like the ridge $\lambda$ sweep from the lesson,
because it is measuring the same trade off along the same axis.

Early stopping here reaches a test error slightly better than the tuned ridge fit, and it
costs nothing extra to compute. That is why it is standard practice in deep learning, where
a full $\lambda$ sweep would mean retraining an expensive model many times. The catch is
that the amount of regularisation now depends on the learning rate, the initialisation and
the optimiser, so it is much harder to reason about than a number you set deliberately.

---
# Exercise 3 — The one standard error rule

> Implement the rule of choosing the largest $\lambda$ whose mean cross validation score is
> within one standard error of the best. Compare against the plain minimum.

The motivation is that cross validation scores are estimates, not measurements. If two
values of $\lambda$ score within the noise of each other, the evidence cannot distinguish
them, and the more heavily regularised of the two is the safer choice. Picking the exact
minimum means fitting the model to the noise in the validation folds.

The standard error of a mean over $k$ folds is $s/\sqrt{k}$, where $s$ is the sample
standard deviation of the fold scores.

In [ ]:
lambdas = np.logspace(-5, 2, 30)
cv_means, cv_errors = [], []

for lam in lambdas:
    fold_scores = []
    for train_idx, val_idx in k_fold_indices(len(y_train), k=5, seed=0):
        predict, _, _ = fit_polynomial(12, lam, x_train[train_idx], y_train[train_idx])
        fold_scores.append(np.mean((predict(x_train[val_idx]) - y_train[val_idx]) ** 2))
    cv_means.append(np.mean(fold_scores))
    cv_errors.append(np.std(fold_scores, ddof=1) / np.sqrt(len(fold_scores)))

cv_means, cv_errors = np.array(cv_means), np.array(cv_errors)

best = int(np.argmin(cv_means))
threshold = cv_means[best] + cv_errors[best]
within = np.where(cv_means <= threshold)[0]
one_se = int(within[-1])          # the largest lambda that still qualifies

print(f"minimum score      : lambda = {lambdas[best]:.4g}, cv = {cv_means[best]:.4f} "
      f"+/- {cv_errors[best]:.4f}")
print(f"acceptance threshold: {threshold:.4f}")
print(f"one standard error : lambda = {lambdas[one_se]:.4g}, cv = {cv_means[one_se]:.4f}")
print(f"\nnote the standard error is {cv_errors[best] / cv_means[best]:.0%} of the mean itself,")
print("so the curve near its minimum is essentially flat within the noise\n")

print(f"{'rule':>22}{'lambda':>10}{'test MSE':>12}{'||w||':>10}")
for label, index in [("plain minimum", best), ("one standard error", one_se)]:
    predict, w_sel, _ = fit_polynomial(12, lambdas[index], x_train, y_train)
    print(f"{label:>22}{lambdas[index]:>10.4g}"
          f"{np.mean((predict(x_test) - y_test) ** 2):>12.4f}{np.linalg.norm(w_sel):>10.4f}")

In [ ]:
plt.errorbar(lambdas, cv_means, yerr=cv_errors, fmt="o-", capsize=3, ms=4,
             label="5-fold cross validation")
plt.axhline(threshold, color="tab:red", ls="--", lw=1,
            label="one standard error above the best")
plt.axvline(lambdas[best], color="gray", ls=":", label=f"minimum: {lambdas[best]:.3g}")
plt.axvline(lambdas[one_se], color="tab:green", ls="-.", label=f"one SE rule: {lambdas[one_se]:.3g}")
plt.xscale("log"); plt.yscale("log")
plt.xlabel("$\\lambda$"); plt.ylabel("validation MSE"); plt.legend(fontsize=8)
plt.title("Everything under the red line is statistically indistinguishable")
plt.show()

The rule picks a larger $\lambda$, giving a smaller weight norm at a slightly worse test
error on this particular split. That is the trade the rule is designed to make: it gives up
a little expected performance in exchange for a model that is less likely to have been
chosen by luck.

The honest reading of this experiment is that the effect is small, because with 20 training
points and 5 folds each validation fold holds only 4 points. The standard error is over half
the mean, so the curve is flat within the noise across a wide range of $\lambda$. The rule
matters far more when folds are larger and the minimum is genuinely sharp, and the wide
error bars here are themselves the argument for not trusting the exact location of the
minimum.

---
# Exercise 4 — Lasso on genuinely irrelevant features

> Build a dataset with 5 informative features and 45 pure noise features. Report how many
> noise weights each method drives to zero, which predicts better, and whether the features
> lasso kept are the informative ones.

This is the situation lasso was designed for. The truth is genuinely sparse: 45 of the 50
features have a true coefficient of exactly zero, and the model has no way of knowing which.

In [ ]:
rng_sparse = np.random.default_rng(11)
n_examples, n_informative, n_noise = 100, 5, 45
X_sparse = rng_sparse.normal(size=(n_examples, n_informative + n_noise))

w_true = np.zeros(n_informative + n_noise)
w_true[:n_informative] = [2.5, -1.8, 1.2, -0.9, 0.6]     # the rest are exactly zero
y_sparse = X_sparse @ w_true + 3.0 + rng_sparse.normal(0, 0.5, n_examples)

X_tr, X_te, y_tr, y_te = train_test_split(X_sparse, y_sparse, 0.4, seed=3)
X_tr_n, mu_s, sigma_s = zscore_normalize(X_tr)
X_te_n, _, _ = zscore_normalize(X_te, mu_s, sigma_s)
print(f"{X_tr.shape[0]} training rows, {X_te.shape[0]} test rows, "
      f"{n_informative} informative features and {n_noise} pure noise")

print(f"\n{'lambda':>8}{'ridge zeroed':>14}{'lasso zeroed':>14}"
      f"{'ridge test':>12}{'lasso test':>12}{'kept all 5?':>13}")
results = {}
for lam in (0.1, 1.0, 5.0, 20.0, 50.0):
    w_r, b_r = ridge_normal_equation(X_tr_n, y_tr, lam)
    w_l, b_l, _ = lasso_gradient_descent(X_tr_n, y_tr, np.zeros(50), 0.0, 0.05, 40_000, lam)
    results[lam] = (w_r, w_l)

    zeroed_r = int(np.sum(np.abs(w_r[n_informative:]) < 1e-8))
    zeroed_l = int(np.sum(np.abs(w_l[n_informative:]) < 1e-8))
    test_r = np.mean((X_te_n @ w_r + b_r - y_te) ** 2)
    test_l = np.mean((X_te_n @ w_l + b_l - y_te) ** 2)
    kept_all = bool(np.all(np.abs(w_l[:n_informative]) > 1e-8))
    print(f"{lam:>8}{zeroed_r:>10} / 45{zeroed_l:>10} / 45"
          f"{test_r:>12.4f}{test_l:>12.4f}{str(kept_all):>13}")

**Ridge zeroes nothing, ever.** All 45 noise features keep a small but nonzero weight at
every $\lambda$, so every one of them contributes noise to every prediction.

**Lasso zeroes almost all of them**, and at $\lambda = 5$ it drives 40 of the 45 noise
weights to exactly zero while keeping all 5 informative features alive. Its test error there
is roughly three times better than the best ridge achieves anywhere.

**Pushed too far it starts deleting real signal.** At $\lambda = 20$ all 45 noise features
are gone, but so are some informative ones, and the test error climbs again. The sparsity
count alone is not a quality measure.

In [ ]:
best_lam = 5.0
w_ridge_best, w_lasso_best = results[best_lam]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
positions = np.arange(50)
for ax, w_shown, name in [(axes[0], w_ridge_best, "ridge"), (axes[1], w_lasso_best, "lasso")]:
    colors = ["tab:red"] * n_informative + ["tab:blue"] * n_noise
    ax.bar(positions, w_shown, color=colors)
    ax.plot(positions, w_true, "k_", ms=9, label="true weight")
    ax.axhline(0, color="gray", lw=0.7)
    survivors = int(np.sum(np.abs(w_shown) > 1e-8))
    ax.set_xlabel("feature index (red = informative, blue = noise)")
    ax.set_ylabel("fitted weight")
    ax.set_title(f"{name} at $\\lambda$ = {best_lam}: {survivors} of 50 weights survive")
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

kept = [int(j) for j in np.where(np.abs(w_lasso_best) > 1e-8)[0]]
print(f"lasso kept features {kept}")
print(f"the informative ones are {list(range(n_informative))}")
print(f"all 5 informative features retained : {set(range(n_informative)).issubset(set(kept))}")
print(f"noise features wrongly retained     : {len([k for k in kept if k >= n_informative])}")

The bar chart makes the difference visible at a glance. Ridge produces 50 small nonzero
bars, a haze of noise features each contributing a little. Lasso produces a handful of bars
that mostly line up with the true weights, and flat zero everywhere else.

This is also the case where the lesson's guidance table earns its keep. Lasso wins decisively
here because the truth really is sparse. On a problem where all 50 features genuinely
mattered a little, ridge would be the better choice, and lasso's habit of picking one member
of a correlated group and deleting the rest would be actively harmful.

---
# Exercise 5 — Learning curves

> Plot training and test error against the number of training examples, for degree 1 and
> degree 15. Explain how the gap tells you whether more data will help.

A learning curve differs from everything else in this lesson: the model is held fixed and
the *dataset size* is the variable. It answers a question you cannot answer any other way,
namely whether collecting more data is worth the money.

In [ ]:
def learning_curve(degree, sizes, repeats=40):
    train_scores, test_scores = [], []
    for size in sizes:
        train_run, test_run = [], []
        for r in range(repeats):
            rng_lc = np.random.default_rng(500 + r)
            x_fit = rng_lc.uniform(-1, 1, size)
            y_fit = truth(x_fit) + rng_lc.normal(0, NOISE, size)
            x_out = rng_lc.uniform(-1, 1, 200)
            y_out = truth(x_out) + rng_lc.normal(0, NOISE, 200)

            predict, _, _ = fit_polynomial(degree, 1e-9, x_fit, y_fit)
            train_run.append(np.mean((predict(x_fit) - y_fit) ** 2))
            test_run.append(np.mean((predict(x_out) - y_out) ** 2))
        train_scores.append(np.mean(train_run))
        # the median resists the occasional catastrophic fit at small sizes
        test_scores.append(np.median(test_run))
    return np.array(train_scores), np.array(test_scores)


sizes = [20, 30, 40, 60, 80, 120, 200, 300, 400]
curves = {d: learning_curve(d, sizes) for d in (1, 15)}

for degree in (1, 15):
    train_scores, test_scores = curves[degree]
    print(f"degree {degree}")
    print(f"{'m':>8}{'train MSE':>12}{'test MSE':>12}{'gap':>12}")
    for size, tr, te in zip(sizes, train_scores, test_scores):
        print(f"{size:>8}{tr:>12.4f}{te:>12.4f}{te - tr:>12.4f}")
    print()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, degree, title in [(axes[0], 1, "degree 1: high bias"),
                          (axes[1], 15, "degree 15: high variance")]:
    train_scores, test_scores = curves[degree]
    ax.plot(sizes, train_scores, "o-", label="training error")
    ax.plot(sizes, test_scores, "s-", label="test error")
    ax.axhline(NOISE ** 2, color="r", ls="--", lw=1, label="irreducible noise")
    ax.set_yscale("log"); ax.set_xlabel("training set size $m$"); ax.legend(fontsize=8)
    ax.set_title(title)
axes[0].set_ylabel("MSE")
plt.tight_layout(); plt.show()

## How to read a learning curve

**Degree 1, on the left.** The two curves meet almost immediately and then run flat
together, well above the noise line. Training error is *high*, which is the giveaway. The
model cannot even fit the data it has already been given, so handing it more of the same
data changes nothing. This is high bias, and the gap between the curves is already closed,
so there is nothing left for extra data to close.

**More data will not help. Use a more flexible model.**

**Degree 15, on the right.** At 20 training points the gap is enormous, with training error
far below test error. The model is memorising. As $m$ grows, training error *rises* and test
error falls steeply, and by 400 points they have nearly met just above the noise floor. This
is high variance, and it is exactly what extra data cures.

**More data will help. Keep collecting, or regularise in the meantime.**

The diagnostic is one question with two follow ups:

| observation | diagnosis | what to do |
|---|---|---|
| both curves high, gap closed | high bias, underfitting | more features, higher degree, less regularisation |
| large gap, training error low | high variance, overfitting | more data, fewer features, more regularisation |
| both curves near the noise floor | as good as it gets | stop |

Notice that training error *increasing* with more data is a healthy sign rather than a
problem. Twenty points can be memorised, four hundred cannot, so the training error rises to
meet the honest level of difficulty of the task.

---
## Recap

| exercise | the transferable lesson |
|---|---|
| 1 | Ridge is least squares plus invented examples that argue for zero weights. This explains the conditioning fix and the Bayesian reading in one identity. |
| 2 | Gradient descent grows the weight norm from zero, so the iteration count is itself a weight budget. Early stopping regularises whether you intended it or not. |
| 3 | Cross validation scores are estimates. When the curve is flat within its own error bars, prefer the simpler model. |
| 4 | When the truth really is sparse, lasso deletes the irrelevant features and wins outright. Pushed too far it deletes real signal too. |
| 5 | Learning curves tell you whether more data is worth buying. High bias shows a closed gap, high variance shows an open one. |

On to lesson 04, neural networks.